# HHS Healthcare Breach Data: Analysis

This notebook answers five questions about healthcare data breaches reported to the U.S. Department of Health and Human Services (HHS). It works from the clean table built by `01_cleaning.ipynb`: 8,013 breaches, each affecting 500 or more people, reported between October 2009 and September 2026.

1. **Trend**: Are breaches becoming more common, and are they affecting more people?
2. **Cause**: Did hacking replace theft and loss as the main cause?
3. **Vendors**: How often are outside vendors (business associates) involved?
4. **Location**: Where was the breached information, and which locations produce the largest breaches?
5. **States**: Which states report the most breaches per resident, and how does New York compare?

Three rules apply throughout:

- **Full years for trends.** 2009 and 2026 are only partly covered, so anything shown per year uses 2010–2025.
- **Medians, not averages.** A few breaches affected tens of millions of people, which pulls an average far above a typical breach. The median (the middle value) describes a typical breach better.
- **With and without mega-breaches.** Results about the number of people affected are shown twice: for all breaches, and leaving out the mega-breaches of 1,000,000+ people. That way a single huge event can't drive a conclusion on its own.

Every chart is interactive (hover for exact values) and is also saved as an HTML file in `reports/figures/`.

## Setup

This loads the clean table and sets up a few shared tools: the 2010–2025 subset used for trends, readable names for the breach type and location columns, and one consistent chart style. If the clean file is missing, the notebook stops and asks for `01_cleaning.ipynb` to be run first.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Interactive charts in JupyterLab and VS Code, plus an HTML version for classic Jupyter and nbviewer
pio.renderers.default = "plotly_mimetype+notebook_connected"

CLEAN_PATH = Path("../data/clean/breaches_clean.csv")
POPULATION_PATH = Path("../data/raw/state_population.csv")
FIGURES_DIR = Path("../reports/figures")
FIRST_YEAR, LAST_YEAR = 2010, 2025  # full calendar years only: 2009 and 2026 are partial

if not CLEAN_PATH.exists():
    raise FileNotFoundError(f"{CLEAN_PATH} is missing. Rerun notebooks/01_cleaning.ipynb first to create it.")

breaches = pd.read_csv(
    CLEAN_PATH,
    parse_dates=["breach_submission_date"],
    dtype={"individuals_affected": "Int64", "year": "Int64", "quarter": "Int64"},
)
full_years = breaches[breaches["year"].between(FIRST_YEAR, LAST_YEAR)]
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TYPE_LABELS = {
    "type_hacking": "Hacking/IT incident",
    "type_theft": "Theft",
    "type_loss": "Loss",
    "type_unauthorized_access": "Unauthorized access/disclosure",
    "type_improper_disposal": "Improper disposal",
    "type_other": "Other",
    "type_unknown": "Unknown",
}
LOCATION_LABELS = {
    "loc_network_server": "Network server",
    "loc_email": "Email",
    "loc_paper_films": "Paper/films",
    "loc_laptop": "Laptop",
    "loc_electronic_medical_record": "Electronic medical record",
    "loc_other": "Other",
    "loc_desktop_computer": "Desktop computer",
    "loc_other_portable_electronic_device": "Other portable device",
}

# Chart colors: a colorblind-safe palette used in a fixed order, plus neutral grays for context and chrome
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
CONTEXT_GRAY, MUTED_GRAY = "#c3c2b7", "#898781"
INK, INK_SECONDARY, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e1e0d9"


def style(fig, title, subtitle, height=520, legend=True):
    """Shared look: left-aligned title and subtitle, legend under them, quiet grid, readable hover labels."""
    fig.update_layout(
        height=height,
        margin=dict(l=70, r=60, t=150 if legend else 115, b=50),
        title=dict(
            text=f"<b>{title}</b>", subtitle=dict(text=subtitle, font=dict(size=13, color=INK_SECONDARY)),
            x=0.01, xanchor="left", xref="container", y=1 - 20 / height, yanchor="top", yref="container",
        ),
        legend=dict(
            orientation="h", title_text="", x=0.01, xanchor="left", xref="container",
            y=1 - 82 / height, yanchor="top", yref="container",
        ),
        font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif', size=13, color=INK),
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        hoverlabel=dict(bgcolor="white", bordercolor=GRID, font_color=INK),
    )
    fig.update_xaxes(showgrid=False, linecolor=CONTEXT_GRAY, tickfont_color=INK_SECONDARY)
    fig.update_yaxes(gridcolor=GRID, gridwidth=1, zeroline=False, tickfont_color=INK_SECONDARY)
    fig.update_annotations(font=dict(size=13, color=INK), selector=dict(xref="paper", yref="paper"))  # subplot titles
    return fig


def label(fig, x, y, text, row=None, col=None, **kwargs):
    """A direct label in secondary ink (never the series color)."""
    fig.add_annotation(x=x, y=y, text=text, showarrow=False, font=dict(size=12, color=INK_SECONDARY),
                       row=row, col=col, **kwargs)


def save_and_show(fig, name):
    """Save the chart as a standalone HTML file in reports/figures/ and display it here."""
    path = FIGURES_DIR / f"{name}.html"
    fig.write_html(path, include_plotlyjs="cdn", config={"displaylogo": False}, div_id=name)  # fixed id: stable reruns
    print(f"Saved {path}")
    fig.show(config={"displaylogo": False})


def ordinal(n):
    return f"{n}{'th' if 11 <= n % 100 <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


print(f"{len(breaches):,} breaches, {breaches['breach_submission_date'].min():%Y-%m-%d} to "
      f"{breaches['breach_submission_date'].max():%Y-%m-%d}; {len(full_years):,} of them in {FIRST_YEAR}-{LAST_YEAR}")
print(f"Mega-breaches (1,000,000+ people): {breaches['mega_breach'].sum()} in total, "
      f"{full_years['mega_breach'].sum()} in {FIRST_YEAR}-{LAST_YEAR}")

8,013 breaches, 2009-10-21 to 2026-09-21; 7,429 of them in 2010-2025
Mega-breaches (1,000,000+ people): 152 in total, 137 in 2010-2025


## 1. Trend: breaches and people affected per year

Counting breaches per year shows whether breaches are becoming more common, and adding up the people affected shows whether they're getting bigger. The people total is shown twice, because a single breach can dominate a year: Change Healthcare alone affected 192.7 million people in 2024. The bottom panel leaves out mega-breaches, so the underlying trend stays visible.

In [2]:
yearly = full_years.groupby("year").agg(
    breaches=("entity_name", "size"),
    people_all=("individuals_affected", "sum"),
    mega_breaches=("mega_breach", "sum"),
    median_people=("individuals_affected", "median"),
)
yearly["people_excl_mega"] = full_years[~full_years["mega_breach"]].groupby("year")["individuals_affected"].sum()
yearly["people_all_m"] = yearly["people_all"] / 1e6
yearly["people_excl_mega_m"] = yearly["people_excl_mega"] / 1e6

print(yearly[["breaches", "people_all_m", "people_excl_mega_m", "mega_breaches", "median_people"]].round(1).to_string())

count, people_all, people_excl, median = (yearly[c] for c in ("breaches", "people_all_m", "people_excl_mega_m", "median_people"))
peak_all, peak_excl = people_all.idxmax(), people_excl.idxmax()
print(f"\nBreaches: {count[FIRST_YEAR]} in {FIRST_YEAR} -> {count[LAST_YEAR]} in {LAST_YEAR} "
      f"({count[LAST_YEAR] / count[FIRST_YEAR]:.1f}x)")
print(f"People, all breaches: {people_all[FIRST_YEAR]:.1f}M -> {people_all[LAST_YEAR]:.1f}M; "
      f"peak {people_all[peak_all]:.1f}M in {peak_all}")
print(f"People, excluding mega-breaches: {people_excl[FIRST_YEAR]:.1f}M -> {people_excl[LAST_YEAR]:.1f}M; "
      f"peak {people_excl[peak_excl]:.1f}M in {peak_excl}")
print(f"Median people per breach: {median[FIRST_YEAR]:,.0f} -> {median[LAST_YEAR]:,.0f}")

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=("Breaches reported", "People affected, all breaches (millions)",
                    "People affected, excluding mega-breaches (millions)"),
)
panels = [
    ("breaches", BLUE, "{:,.0f}", "%{x}: %{y:,} breaches"),
    ("people_all_m", BLUE, "{:.1f}M", "%{x}: %{y:.1f} million people"),
    ("people_excl_mega_m", ORANGE, "{:.1f}M", "%{x}: %{y:.1f} million people"),
]
for row, (column, color, fmt, hover) in enumerate(panels, start=1):
    values = yearly[column]
    labelled = {FIRST_YEAR, LAST_YEAR, values.idxmax()}  # label the first year, the last year and the peak
    fig.add_trace(go.Bar(
        x=yearly.index, y=values, marker=dict(color=color, cornerradius=4), showlegend=False,
        text=[fmt.format(v) if year in labelled else "" for year, v in values.items()],
        textposition="outside", constraintext="none", textfont=dict(color=INK_SECONDARY, size=12), cliponaxis=False,
        hovertemplate=hover + "<extra></extra>",
    ), row=row, col=1)
    fig.update_yaxes(range=[0, values.max() * 1.25], tickformat="~s", row=row, col=1)

style(fig, f"Reported breaches rose from {count[FIRST_YEAR]} in {FIRST_YEAR} to {count[LAST_YEAR]} in {LAST_YEAR}",
      "People affected rose too, with and without mega-breaches (1,000,000+ people)", height=800, legend=False)
fig.update_layout(bargap=0.5)
fig.update_xaxes(dtick=1, tickangle=0)
save_and_show(fig, "q1_breaches_per_year")

      breaches  people_all_m  people_excl_mega_m  mega_breaches  median_people
year                                                                          
2010       199           5.9                 3.7              2         2027.0
2011       200          13.2                 3.6              4         2270.5
2012       218           2.9                 2.9              0         2332.5
2013       277           7.0                 3.0              1         2352.0
2014       314          19.1                 5.4              4         2674.0
2015       270         112.5                 4.0              6         2030.0
2016       328          16.7                 7.4              3         2380.0
2017       358           5.3                 5.3              0         2075.0
2018       369          15.3                 8.9              4         2393.0
2019       511          45.0                14.6              7         4000.0
2020       663          35.3                25.3    

### Finding

Reported breaches roughly quadrupled, from 199 in 2010 to 803 in 2025. People affected grew much faster, from 5.9 million in 2010 to a peak of 290.1 million in 2024, the year of the Change Healthcare breach. Without mega-breaches the rise is smaller but still steep: from 3.7 million to a peak of 45.6 million in 2023 and 28.1 million in 2025, so the growth isn't driven only by a few huge events.

## 2. Cause: how the mix of breach types changed

Each year's breaches are split by type, as the share of that year's breaches involving each type. Shares are used instead of counts because the total number of breaches grew so much. The question is whether hacking replaced the physical causes, theft and loss, and in which year it pulled ahead of them for good.

A breach can be filed under several types (for example, both hacking and theft), so a year's shares can add up to more than 100%. The one breach with no type is left out. No breach after 2014 is filed as "Other" or "Unknown".

In [3]:
known_type = full_years[full_years["type_of_breach"].notna()]
type_share = known_type.groupby("year")[list(TYPE_LABELS)].mean().mul(100)

most_common = type_share.idxmax(axis=1).map(TYPE_LABELS)
hacking_ahead = type_share["type_hacking"].gt(type_share["type_theft"]) & type_share["type_hacking"].gt(type_share["type_loss"])
overtake_year = hacking_ahead.index[~hacking_ahead.to_numpy()].max() + 1  # ahead of theft and loss every year since
hacking_top_from = most_common.index[(most_common != TYPE_LABELS["type_hacking"]).to_numpy()].max() + 1

print(type_share.round(1).rename(columns=TYPE_LABELS).to_string())
print("\nMost common type each year:")
print(most_common.to_string())
print(f"\nHacking has been ahead of both theft and loss every year since {overtake_year} "
      f"({type_share.loc[overtake_year, 'type_hacking']:.1f}% vs theft {type_share.loc[overtake_year, 'type_theft']:.1f}% "
      f"and loss {type_share.loc[overtake_year, 'type_loss']:.1f}%), and the most common type since {hacking_top_from}.")
for year in (FIRST_YEAR, LAST_YEAR):
    print(f"{year}: hacking {type_share.loc[year, 'type_hacking']:.1f}%, theft {type_share.loc[year, 'type_theft']:.1f}%, "
          f"loss {type_share.loc[year, 'type_loss']:.1f}%")

# The story series get the palette in fixed order; the minor types are thin gray context lines
EMPHASIS = {"type_hacking": BLUE, "type_theft": ORANGE, "type_loss": AQUA, "type_unauthorized_access": YELLOW}
fig = go.Figure()
for rank, (column, name) in enumerate(TYPE_LABELS.items()):
    emphasized = column in EMPHASIS
    fig.add_trace(go.Scatter(
        x=type_share.index, y=type_share[column], name=name, mode="lines", legendrank=rank,
        line=dict(color=EMPHASIS.get(column, CONTEXT_GRAY), width=2 if emphasized else 1.5),
        hovertemplate=f"{name}: %{{y:.1f}}%<extra></extra>",
    ))
fig.data = tuple(sorted(fig.data, key=lambda trace: trace.line.color != CONTEXT_GRAY))  # draw gray lines underneath

h_first, h_last = type_share.loc[FIRST_YEAR, "type_hacking"], type_share.loc[LAST_YEAR, "type_hacking"]
style(fig, f"Hacking went from {h_first:.0f}% of breaches in {FIRST_YEAR} to {h_last:.0f}% in {LAST_YEAR}",
      "Share of each year's breaches involving each type. A breach can have several types, so shares can add up to more than 100%.")
fig.update_layout(hovermode="x unified")
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.add_vline(x=overtake_year, line=dict(color=MUTED_GRAY, width=1))
label(fig, overtake_year, 97, f"{overtake_year}: hacking passes theft and loss", xanchor="left", xshift=6)
label(fig, LAST_YEAR, h_last, f"{h_last:.0f}%", xanchor="left", xshift=8)
label(fig, FIRST_YEAR, type_share.loc[FIRST_YEAR, "type_theft"], f"Theft {type_share.loc[FIRST_YEAR, 'type_theft']:.0f}%",
      xanchor="left", yanchor="bottom", xshift=4, yshift=4)
save_and_show(fig, "q2_breach_types")

      Hacking/IT incident  Theft  Loss  Unauthorized access/disclosure  Improper disposal  Other  Unknown
year                                                                                                     
2010                  4.0   68.2  10.1                             5.1                5.1   11.6      0.0
2011                  8.5   61.5  10.0                            18.0                3.5    1.0      3.5
2012                  7.8   60.6   9.6                            19.7                3.7    9.2      1.4
2013                 10.5   48.4   9.0                            27.1                5.1    7.6      1.8
2014                 12.4   41.1   9.9                            34.1                3.8    8.6      0.3
2015                 20.7   30.0   8.9                            38.1                2.2    0.0      0.0
2016                 34.8   18.6   4.9                            39.6                2.1    0.0      0.0
2017                 41.6   15.4   4.5        

### Finding

Yes. Theft was the most common type every year from 2010 to 2014 (68.2% of breaches in 2010). Hacking pulled ahead of both theft and loss in 2016 (34.8% vs 18.6% and 4.9%) and has stayed ahead every year since; it became the single most common type in 2017, after unauthorized access/disclosure led in 2015–2016. By 2025, hacking was involved in 80.8% of breaches, theft in 1.1% and loss in 0.2%.

## 3. Vendors: how often a business associate was involved

Healthcare organizations share patient data with outside vendors, called business associates, such as billing companies, IT providers and claims processors. A breach counts as vendor-related if the portal says a business associate was present, or if a business associate filed the report itself.

For each year, the chart shows the share of breaches that were vendor-related (left) and the share of all people affected that those breaches account for (right). The people share is shown with and without mega-breaches, because one huge vendor breach can swing it. To answer whether the share has grown, the first five full years (2010–2014) are compared with the last five (2021–2025), each pooled into a single figure.

In [4]:
vendor_data = full_years.assign(
    vendor=full_years["business_associate_present"] | full_years["covered_entity_type"].eq("Business Associate")
)


def vendor_shares(df):
    """Share (%) of breaches, and of people affected, that involved a vendor."""
    non_mega = df[~df["mega_breach"]]
    return pd.Series({
        "% of breaches": df["vendor"].mean() * 100,
        "% of people (all breaches)": df.loc[df["vendor"], "individuals_affected"].sum() / df["individuals_affected"].sum() * 100,
        "% of people (excl. mega-breaches)":
            non_mega.loc[non_mega["vendor"], "individuals_affected"].sum() / non_mega["individuals_affected"].sum() * 100,
    })


vendor_by_year = pd.DataFrame({year: vendor_shares(group) for year, group in vendor_data.groupby("year")}).T
periods = {"2010-2014": (2010, 2014), "2021-2025": (2021, 2025)}
pooled = pd.DataFrame({name: vendor_shares(vendor_data[vendor_data["year"].between(*span)]) for name, span in periods.items()}).T

print(vendor_by_year.round(1).to_string())
print("\nPooled over five years:")
print(pooled.round(1).to_string())

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.08,
                    subplot_titles=("Share of breaches", "Share of people affected"))
series = [
    (1, "% of breaches", "All breaches", BLUE, False),
    (2, "% of people (all breaches)", "All breaches", BLUE, True),
    (2, "% of people (excl. mega-breaches)", "Excluding mega-breaches", ORANGE, True),
]
for col, column, name, color, in_legend in series:
    values = vendor_by_year[column]
    fig.add_trace(go.Scatter(
        x=values.index, y=values, name=name, mode="lines", line=dict(color=color, width=2),
        legendgroup=name, showlegend=in_legend, hovertemplate=f"{name}: %{{y:.1f}}%<extra></extra>",
    ), row=1, col=col)
    fig.add_trace(go.Scatter(  # end dot
        x=[LAST_YEAR], y=[values[LAST_YEAR]], mode="markers", showlegend=False, hoverinfo="skip",
        marker=dict(color=color, size=8, line=dict(color=SURFACE, width=2)),
    ), row=1, col=col)

first, last = pooled.loc["2010-2014", "% of breaches"], pooled.loc["2021-2025", "% of breaches"]
style(fig, f"Vendors were involved in {last:.0f}% of breaches in 2021–2025, up from {first:.0f}% in 2010–2014",
      "Share of each year's breaches and people affected that involved a business associate")
for col, column, *_ in series:
    end_value = vendor_by_year.loc[LAST_YEAR, column]
    label(fig, LAST_YEAR, end_value, f"{end_value:.0f}%", row=1, col=col, xanchor="left", xshift=8)
fig.update_layout(hovermode="x unified")
fig.update_xaxes(dtick=3)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
save_and_show(fig, "q3_vendor_share")

      % of breaches  % of people (all breaches)  % of people (excl. mega-breaches)
2010           23.1                        33.9                               54.5
2011           22.5                        67.9                               12.1
2012           18.3                        40.2                               40.2
2013           24.2                        15.3                               35.9
2014           25.8                        46.8                               45.0
2015           16.7                         4.8                               48.3
2016           20.7                        30.6                               22.2
2017           12.8                         6.9                                6.9
2018           24.4                        48.7                               26.2
2019           23.3                        57.0                               26.7
2020           39.8                        63.4                               57.0
2021

### Finding

Yes, by count. Vendor-related breaches rose from 23.1% of all breaches in 2010–2014 to 34.7% in 2021–2025, and have been above 30% every year since 2020. Their share of people affected also rose, from 46.0% to 66.9%, but that came from a few huge vendor breaches: without mega-breaches, the share slipped from 38.3% to 32.9%.

## 4. Location: where the breached information was

Each breach records where the exposed information was: a network server, email, paper records, a laptop and so on. For each location, this counts the breaches and finds the median number of people affected, to show which location produces the most breaches and which produces the largest ones.

This isn't a trend over time, so it uses all 8,013 breaches. A breach can involve several locations, so it's counted under each of them, and the counts add up to more than the total.

In [5]:
location_stats = pd.DataFrame([
    {
        "location": name,
        "breaches": int(breaches[column].sum()),
        "share_of_breaches_%": breaches[column].mean() * 100,
        "median_people_all": breaches.loc[breaches[column], "individuals_affected"].median(),
        "median_people_excl_mega": breaches.loc[breaches[column] & ~breaches["mega_breach"], "individuals_affected"].median(),
        "mega_breaches": int((breaches[column] & breaches["mega_breach"]).sum()),
    }
    for column, name in LOCATION_LABELS.items()
]).sort_values("breaches", ascending=False, ignore_index=True)
print(location_stats.round(1).to_string(index=False))

most_common = location_stats.iloc[0]
largest = location_stats.loc[location_stats["median_people_all"].idxmax()]
print(f"\nMost breaches: {most_common['location']} ({most_common['breaches']:,}, "
      f"{most_common['share_of_breaches_%']:.0f}% of all breaches)")
print(f"Largest breaches: {largest['location']}, median {largest['median_people_all']:,.0f} people "
      f"({largest['median_people_excl_mega']:,.0f} excluding mega-breaches); "
      f"{largest['mega_breaches']} of {breaches['mega_breach'].sum()} mega-breaches involved it")

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, column_widths=[0.45, 0.55],
                    subplot_titles=("Number of breaches", "Median people affected per breach"))
fig.add_trace(go.Bar(
    y=location_stats["location"], x=location_stats["breaches"], orientation="h", showlegend=False,
    marker=dict(color=BLUE, cornerradius=4), hovertemplate="%{y}: %{x:,} breaches<extra></extra>",
), row=1, col=1)
for column, name, color in (("median_people_all", "All breaches", BLUE),
                            ("median_people_excl_mega", "Excluding mega-breaches", ORANGE)):
    fig.add_trace(go.Bar(
        y=location_stats["location"], x=location_stats[column], orientation="h", name=name,
        marker=dict(color=color, cornerradius=4), hovertemplate=f"%{{y}}: median %{{x:,.0f}} people ({name.lower()})<extra></extra>",
    ), row=1, col=2)

title = (f"{most_common['location']} breaches are both the most common and the largest"
         if most_common["location"] == largest["location"] else
         f"{most_common['location']} breaches are the most common; {largest['location']} breaches are the largest")
style(fig, title, f"All {len(breaches):,} breaches, {breaches['year'].min()}–{breaches['year'].max()}. "
      "A breach can involve several locations and is counted under each.", height=520)
fig.update_layout(barmode="group", bargap=0.35, bargroupgap=0.1)
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(showgrid=True, gridcolor=GRID, zeroline=False, tickformat="~s")
fig.update_xaxes(range=[0, location_stats["breaches"].max() * 1.2], row=1, col=1)
fig.update_xaxes(range=[0, location_stats["median_people_all"].max() * 1.25], row=1, col=2)
label(fig, most_common["breaches"], most_common["location"], f"{most_common['breaches']:,}", row=1, col=1, xanchor="left", xshift=6)
for column, shift in (("median_people_all", 7), ("median_people_excl_mega", -7)):  # blue bar sits above orange
    label(fig, largest[column], largest["location"], f"{largest[column]:,.0f}", row=1, col=2, xanchor="left", xshift=6, yshift=shift)
save_and_show(fig, "q4_locations")

                 location  breaches  share_of_breaches_%  median_people_all  median_people_excl_mega  mega_breaches
           Network server      3691                 46.1             9967.0                   8962.0            134
                    Email      1855                 23.1             3259.0                   3232.0              8
              Paper/films      1012                 12.6             1500.0                   1500.0              0
                   Laptop       514                  6.4             2292.0                   2277.5              2
Electronic medical record       477                  6.0             2864.0                   2813.0              5
                    Other       462                  5.8             2560.5                   2500.0              9
         Desktop computer       380                  4.7             2367.0                   2354.0              2
    Other portable device       331                  4.1             228

Saved ../reports/figures/q4_locations.html


### Finding

Network servers produce both the most breaches and the largest ones: 3,691 breaches (46% of all) with a median of 9,967 people each, about three times email's median of 3,259. Leaving out mega-breaches barely changes this (median 8,962), and 134 of the 152 mega-breaches involved a network server. Paper records are common (1,012 breaches) but small, with a median of 1,500 people.

## 5. States: breaches per resident, and New York's profile

Big states naturally report more breaches, so states are ranked by **breaches per million residents**, using each state's 2020 Census population (`data/raw/state_population.csv`; the source is in the README). The count covers 2010–2025. New York's breaches are then compared with the national pattern: the share of breaches involving each type and each location.

Caveats to keep in mind:

- A breach is counted in the state of the organization that reported it, not where the affected people live. A large insurer can affect people across the country.
- One population figure, from 2020, is used for all 16 years.
- States with few breaches can move several places because of a handful of reports.
- Puerto Rico and records without a state are left out of the ranking.

In [6]:
# 2020 Census resident population for the 50 states and DC (U.S. Census Bureau; source details in README.md)
population = pd.read_csv(POPULATION_PATH)

state_rates = (
    full_years.loc[full_years["state"].isin(population["state"])]
    .groupby("state").size().rename("breaches")
    .reindex(population["state"].tolist(), fill_value=0).rename_axis("state").reset_index()
    .merge(population, on="state")
)
state_rates["per_million"] = state_rates["breaches"] / state_rates["population_2020"] * 1_000_000
state_rates["rank"] = state_rates["per_million"].rank(ascending=False, method="min").astype(int)
state_rates = state_rates.sort_values("rank", ignore_index=True)
national_rate = state_rates["breaches"].sum() / state_rates["population_2020"].sum() * 1_000_000
ny = state_rates.set_index("state").loc["NY"]

print(f"Left out of the ranking: {full_years['state'].eq('PR').sum()} Puerto Rico breaches, "
      f"{full_years['state'].isna().sum()} with no state")
print(f"All 50 states + DC: {national_rate:.1f} breaches per million residents\n")
print(state_rates[["rank", "state_name", "breaches", "population_2020", "per_million"]].round(1).to_string(index=False))
print(f"\nNew York: {ordinal(ny['rank'])} of {len(state_rates)}, {ny['per_million']:.1f} breaches per million "
      f"({ny['breaches']} breaches, {ny['population_2020']:,} residents)")

flag_labels = {**TYPE_LABELS, **LOCATION_LABELS}
ny_breaches = full_years[full_years["state"].eq("NY")]
pattern = pd.DataFrame({
    "New York": ny_breaches[list(flag_labels)].mean() * 100,
    "All U.S.": full_years[list(flag_labels)].mean() * 100,
})
pattern["Difference (points)"] = pattern["New York"] - pattern["All U.S."]
print(f"\nShare of breaches involving each type and location, New York ({len(ny_breaches)}) vs all U.S. ({len(full_years):,}):")
table_labels = {**{c: f"Type: {n}" for c, n in TYPE_LABELS.items()}, **{c: f"Location: {n}" for c, n in LOCATION_LABELS.items()}}
print(pattern.rename(index=table_labels).round(1).to_string())

fig = make_subplots(
    rows=2, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.2, vertical_spacing=0.12,
    specs=[[{"rowspan": 2}, {}], [None, {}]],
    subplot_titles=(f"Breaches per million residents (line: all states, {national_rate:.1f})", "Breach type", "Breach location"),
)
fig.add_trace(go.Bar(
    x=state_rates["per_million"], y=state_rates["state"], orientation="h", showlegend=False,
    marker=dict(color=[BLUE if state == "NY" else CONTEXT_GRAY for state in state_rates["state"]], cornerradius=3),
    customdata=state_rates[["state_name", "breaches", "rank"]],
    hovertemplate="%{customdata[0]}: %{x:.1f} per million (%{customdata[1]} breaches, rank %{customdata[2]})<extra></extra>",
), row=1, col=1)
fig.add_vline(x=national_rate, line=dict(color=MUTED_GRAY, width=1), row=1, col=1)
fig.update_yaxes(autorange="reversed", showgrid=False, tickfont=dict(size=10), row=1, col=1)
fig.update_xaxes(showgrid=True, gridcolor=GRID, range=[0, state_rates["per_million"].max() * 1.35], row=1, col=1)

# Dumbbells: New York (blue) vs all U.S. (gray), joined by a thin line
for row, columns in ((1, list(TYPE_LABELS)), (2, list(LOCATION_LABELS))):
    part = pattern.loc[columns].sort_values("All U.S.", ascending=False)
    names = [flag_labels[column] for column in part.index]
    xs, ys = [], []
    for name, us_share, ny_share in zip(names, part["All U.S."], part["New York"]):
        xs += [us_share, ny_share, None]
        ys += [name, name, None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=GRID, width=3), hoverinfo="skip",
                             showlegend=False), row=row, col=2)
    for name, color in (("All U.S.", MUTED_GRAY), ("New York", BLUE)):
        fig.add_trace(go.Scatter(
            x=part[name], y=names, mode="markers", name=name, legendgroup=name, showlegend=(row == 1),
            marker=dict(color=color, size=10, line=dict(color=SURFACE, width=2)),
            hovertemplate=f"{name}: %{{x:.1f}}%<extra>%{{y}}</extra>",
        ), row=row, col=2)
    fig.update_yaxes(autorange="reversed", showgrid=False, row=row, col=2)
    fig.update_xaxes(showgrid=True, gridcolor=GRID, ticksuffix="%",
                     range=[0, part[["New York", "All U.S."]].max().max() * 1.1], row=row, col=2)

style(fig, f"New York ranks {ordinal(ny['rank'])} of {len(state_rates)} for breaches per resident",
      f"Left: breaches reported {FIRST_YEAR}–{LAST_YEAR} per million residents (2020 Census). "
      "Right: New York's mix of breach types and locations vs all U.S. breaches", height=1050)
label(fig, ny["per_million"], "NY", f"New York: {ny['per_million']:.1f} ({ordinal(ny['rank'])})", row=1, col=1,
      xanchor="left", xshift=6)
save_and_show(fig, "q5_states")

Left out of the ranking: 32 Puerto Rico breaches, 21 with no state
All 50 states + DC: 22.3 breaches per million residents

 rank           state_name  breaches  population_2020  per_million
    1 District of Columbia        29           689545         42.1
    2               Alaska        30           733391         40.9
    3          Connecticut       136          3605944         37.7
    4         Rhode Island        40          1097379         36.5
    5              Montana        36          1084225         33.2
    6              Wyoming        19           576851         32.9
    7        Massachusetts       226          7029917         32.1
    8            Minnesota       176          5706494         30.8
    9             Delaware        30           989948         30.3
   10             Nebraska        59          1961504         30.1
   11             Arkansas        89          3011524         29.6
   12                 Iowa        92          3190369         28.8
   13

### Finding

New York ranks 26th of 51 (the 50 states and DC) with 24.2 breaches per million residents in 2010–2025, a little above the rate for all states combined (22.3). The highest rates are in DC (42.1), Alaska (40.9) and Connecticut (37.7), and the lowest is Louisiana (13.3). New York's breaches lean slightly more toward hacking (63.7% vs 58.5% nationally) and network servers (48.4% vs 44.2%), and less toward paper records (9.6% vs 13.3%).

## Key Findings

- **Breaches have quadrupled.** Reported healthcare breaches rose from 199 in 2010 to 803 in 2025. People affected per year rose from 5.9 million to 140.6 million, peaking at 290.1 million in 2024; excluding mega-breaches (1 million+ people), the rise was from 3.7 million to 28.1 million.
- **Hacking replaced theft and loss.** Theft was involved in 68% of breaches in 2010. Hacking overtook theft and loss in 2016 and was involved in 81% of breaches by 2025.
- **Vendors are involved more often.** Breaches involving a business associate rose from 23% of all breaches in 2010–2014 to 35% in 2021–2025. Their share of people affected rose from 46% to 67%, but a few huge vendor breaches drove that: excluding mega-breaches, it fell from 38% to 33%.
- **Network servers lead on both count and size.** Of all 8,013 breaches (2009–2026), 3,691 (46%) involved a network server, with a median of 9,967 people affected, about three times the median for email breaches.
- **New York is in the middle.** New York ranks 26th of the 50 states and DC, with 24.2 breaches per million residents in 2010–2025, close to the national rate of 22.3. DC (42.1), Alaska (40.9) and Connecticut (37.7) have the highest rates.